In [1]:
# STEP 1: IMPORT LIBRARIES
import pandas as pd
import numpy as np
import os

folder_path = r"D:\Desktop\Data Analytics\Projects\IKEA-Customer-Analytics"
print(os.listdir(folder_path))

['01_Dataset', '02_PowerBI', '03_SQL', '04_Python', '05_Screenshots', '06_Documentation']


In [2]:

# STEP 2: LOAD AND PREPARE THE DATASET

file_path = r"D:\Desktop\Data Analytics\Projects\IKEA-Customer-Analytics\RetailCleaned.csv"

df = pd.read_csv(file_path, header=None, low_memory=False)

df.columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer_ID",
    "Country",
    "SourceSheet"
]

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"], errors="coerce"
)
df["Quantity"] = pd.to_numeric(
    df["Quantity"], errors="coerce"
)
df["Price"] = pd.to_numeric(
    df["Price"], errors="coerce"
)
df["Customer_ID"] = pd.to_numeric(
    df["Customer_ID"], errors="coerce"
)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Desktop\\Data Analytics\\Projects\\IKEA-Customer-Analytics\\RetailCleaned.csv'

In [ ]:

# STEP 3: VALIDATE THE DATE RANGE

print("Earliest date:", df["InvoiceDate"].min())
print("Latest date:", df["InvoiceDate"].max())


In [ ]:
# STEP 4: CALCULATE RFM METRICS

rfm_data = df[
    (df["Customer_ID"].notna()) &
    (df["Quantity"] > 0)
].copy()

snapshot_date = pd.to_datetime("2011-12-10")

rfm = rfm_data.groupby("Customer_ID").agg(
    Recency=(
        "InvoiceDate",
        lambda x: (snapshot_date - x.max()).days
    ),
    Frequency=("Invoice", "nunique"),
    Monetary=(
        "Invoice",
        lambda x: (
            rfm_data.loc[x.index, "Quantity"] *
            rfm_data.loc[x.index, "Price"]
        ).sum()
    )
).reset_index()

print("RFM calculation completed!")
print("RFM dataset shape:", rfm.shape)
display(rfm.head())

In [ ]:

# STEP 5: SCORE CUSTOMERS AND ASSIGN BUSINESS SEGMENTS

rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

def segment_customer(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal'
    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'
    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New'
    elif row['R_Score'] == 3 and row['F_Score'] <= 2:
        return 'Needs Attention'
    else:
        return 'Lost'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)
rfm['IsChurned'] = rfm['Recency'] > 90


In [ ]:
 # STEP 6: PREPARE AND EXPORT THE FINAL DATASET

df_final = rfm_data.copy()

df_final["InvoiceMonth"] = (
    df_final["InvoiceDate"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

df_final["Revenue"] = (
    df_final["Quantity"] * df_final["Price"]
)

rfm_columns = [
    "Customer_ID",
    "Recency",
    "Frequency",
    "Monetary",
    "R_Score",
    "F_Score",
    "M_Score",
    "Segment",
    "IsChurned"
]

df_final = df_final.merge(
    rfm[rfm_columns],
    on="Customer_ID",
    how="left",
    validate="many_to_one"
)

output_path = os.path.join(
    folder_path,
    "EcommerceDashboardData.xlsx"
)

df_final.to_excel(output_path, index=False)

print("Export completed successfully!")
print("Final dataset shape:", df_final.shape)
print("Saved to:", output_path)

In [ ]:
# STEP 7: VALIDATE THE FINAL DATASET

print("Dataset shape:", df_final.shape)
print("\nColumns:")
print(df_final.columns.tolist())

display(df_final.head())

In [ ]:
# STEP 8: CUSTOMER SEGMENT SUMMARY

segment_counts = rfm["Segment"].value_counts()

print("Total unique customers:", rfm["Customer_ID"].nunique())
print("\nCustomers by segment:")
print(segment_counts)